# Bird Species Classification — Results & Analysis

Graphs for the dataset, training progress and test-set performance.

**Workflow**
1. Train in a terminal (a long run is safer there than inside a notebook kernel):
   `python train.py --train-dir data/Train --train-txt data/train.txt --epochs 30 --workers 0`
2. Run this notebook top to bottom (*Run All*). Every figure is also saved to `figures/` for the report.

Section 1 works before training; sections 2–3 need `checkpoints/history.csv` and `checkpoints/best.pt`.

In [ ]:
import os, re, random, collections

import numpy as np
import pandas as pd
import torch
import timm
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader

from src.data import BirdDataset, build_transforms, accuracy_metrics, read_split_file, find_image

CKPT = "checkpoints/best.pt"
HISTORY = "checkpoints/history.csv"
TRAIN_DIR, TRAIN_TXT = "data/Train", "data/train.txt"
TEST_DIR, TEST_TXT = "data/Test", "data/test.txt"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# Chart style: one recessive grid, thin marks, a small fixed colour set
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUES = LinearSegmentedColormap.from_list(
    "blues", [SURFACE, "#cde2fb", "#86b6ef", "#2a78d6", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
})

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches="tight")

pct = mtick.PercentFormatter(1.0, decimals=0)

## 1. The dataset

Species names aren't in the label files, but the training filenames contain them
(`Black_footed_Albatross_0004_2731401028.jpg` → *Black footed Albatross*).
This also writes `classes.txt`, which the web app uses to show names instead of numbers.

In [ ]:
def species_from_filename(fname):
    m = re.match(r"^(.*?)_\d+_\d+\.\w+$", os.path.basename(fname))
    return m.group(1) if m else os.path.splitext(fname)[0]

train_raw = read_split_file(TRAIN_TXT)
test_raw = read_split_file(TEST_TXT)
offset = min(l for _, l in train_raw)

votes = collections.defaultdict(collections.Counter)
for name, label in train_raw:
    votes[label - offset][species_from_filename(name)] += 1
num_classes = max(votes) + 1
CLASS_NAMES = [votes[i].most_common(1)[0][0].replace("_", " ") for i in range(num_classes)]
FAMILY = [n.split()[-1].capitalize() for n in CLASS_NAMES]  # "Cape May Warbler" -> "Warbler"

with open("classes.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(CLASS_NAMES) + "\n")

train_counts = np.bincount([l - offset for _, l in train_raw], minlength=num_classes)
test_counts = np.bincount([l - offset for _, l in test_raw], minlength=num_classes)
display(Markdown(
    f"**{num_classes} species** · **{len(train_raw):,} training images** "
    f"({train_counts.min()}–{train_counts.max()} per species, mean {train_counts.mean():.1f}) · "
    f"**{len(test_raw):,} test images** ({test_counts.min()}–{test_counts.max()} per species)"))

In [ ]:
# How many training photos each species has
fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.arange(train_counts.min(), train_counts.max() + 2) - 0.5
ax.hist(train_counts, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=2)
ax.set(title="Training images per species", xlabel="Images", ylabel="Number of species")
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
save(fig, "01_images_per_species.png"); plt.show()

In [ ]:
# Bird families with the most look-alike species (the hardest part of the task)
fam = pd.Series(FAMILY).value_counts()
fam = fam[fam >= 4]
fig, ax = plt.subplots(figsize=(8, 0.32 * len(fam) + 0.8))
ax.barh(fam.index[::-1], fam.values[::-1], color=BLUE, height=0.7)
for y, v in enumerate(fam.values[::-1]):
    ax.text(v + 0.2, y, str(v), va="center", color=INK2, fontsize=9)
ax.set(title="Families with 4+ species in the dataset", xlabel="Number of species")
ax.grid(axis="y", visible=False)
save(fig, "02_families.png"); plt.show()

In [ ]:
# A random look at the training images
random.seed(1)
picks = random.sample(train_raw, 10)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, (name, label) in zip(axes.flat, picks):
    ax.imshow(Image.open(find_image(TRAIN_DIR, name)).convert("RGB"))
    ax.set_title(CLASS_NAMES[label - offset], fontsize=9, fontweight="normal", loc="center")
    ax.axis("off")
fig.tight_layout()
save(fig, "03_sample_images.png"); plt.show()

## 2. Training progress (learning curves)

How to read these:
- **Loss going down** = the model is learning.
- **Validation accuracy flattening out** = more epochs won't help much.
- **Train accuracy is lower than validation early on** — that's expected here: training images are
  randomly cropped/flipped/recoloured (harder), and label smoothing caps training confidence.
- **Overfitting** would look like validation accuracy *falling* while training keeps improving.

In [ ]:
hist = pd.read_csv(HISTORY)
best = hist.loc[hist.val_avg_class.idxmax()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
ax1.plot(hist.epoch, hist.train_loss, color=BLUE)
ax1.set(title="Training loss", xlabel="Epoch", ylabel="Loss")

for col, label, color in [("train_acc", "Train", BLUE),
                          ("val_top1", "Validation top-1", ORANGE),
                          ("val_avg_class", "Validation avg per class", AQUA)]:
    ax2.plot(hist.epoch, hist[col], color=color, label=label)
ax2.axvline(best.epoch, color=MUTED, linestyle="--", linewidth=1)
ax2.annotate(f"best: epoch {int(best.epoch)}\n{best.val_avg_class:.1%}",
             (best.epoch, best.val_avg_class), xytext=(-8, -30), textcoords="offset points",
             ha="right", color=INK2, fontsize=9)
ax2.yaxis.set_major_formatter(pct)
ax2.set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
ax2.legend(loc="lower right")
for ax in (ax1, ax2):
    ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
fig.tight_layout()
save(fig, "04_learning_curves.png"); plt.show()

print(f"Best checkpoint: epoch {int(best.epoch)} — val top-1 {best.val_top1:.2%}, "
      f"val avg/class {best.val_avg_class:.2%}")
print(f"Total training time: {hist.time_s.sum() / 60:.1f} min")

## 3. Test-set results

The test set is only touched here, after training is finished.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, map_location="cpu")
model = timm.create_model(ckpt["model_name"], pretrained=False, num_classes=ckpt["num_classes"])
model.load_state_dict(ckpt["state_dict"])
model.to(device).eval()

_, eval_tf = build_transforms(ckpt["img_size"])
test_ds = BirdDataset(TEST_DIR, TEST_TXT, transform=eval_tf, label_offset=ckpt["label_offset"])
loader = DataLoader(test_ds, batch_size=64, num_workers=0)

probs, targets = [], []
with torch.no_grad():
    for x, y in loader:
        probs.append(model(x.to(device)).softmax(1).cpu())
        targets.append(y)
probs, targets = torch.cat(probs), torch.cat(targets)
preds = probs.argmax(1)
conf = probs.max(1).values

top1, avg_cls, _ = accuracy_metrics(preds, targets, num_classes)
top5 = (probs.topk(5, dim=1).indices == targets[:, None]).any(1).float().mean().item()
display(Markdown(
    f"| Metric | Test score |\n|---|---|\n"
    f"| **Top-1 accuracy** | **{top1:.2%}** |\n"
    f"| **Average accuracy per class** | **{avg_cls:.2%}** |\n"
    f"| Top-5 accuracy (right answer in the top 5 guesses) | {top5:.2%} |\n\n"
    f"Model: `{ckpt['model_name']}` · {len(test_ds)} test images"))

In [ ]:
# Per-species results table (also saved as CSV for the report)
t, p = targets.numpy(), preds.numpy()
per = pd.DataFrame({"label": np.arange(num_classes) + offset, "species": CLASS_NAMES, "family": FAMILY})
per["n_test"] = np.bincount(t, minlength=num_classes)
per["correct"] = np.bincount(t, weights=(p == t), minlength=num_classes).astype(int)
per["accuracy"] = per.correct / per.n_test.replace(0, np.nan)
per.sort_values("accuracy").to_csv(os.path.join(FIG_DIR, "per_class_accuracy.csv"), index=False)

perfect = (per.accuracy == 1).sum()
display(Markdown(f"**{perfect}** of {num_classes} species are classified perfectly; "
                 f"**{(per.accuracy < 0.5).sum()}** are below 50%."))

In [ ]:
# How accuracy is spread across the 200 species
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(per.accuracy.dropna(), bins=np.linspace(0, 1, 11), color=BLUE, edgecolor=SURFACE, linewidth=2)
ax.axvline(avg_cls, color=INK2, linestyle="--", linewidth=1)
right = avg_cls > 0.7
ax.text(avg_cls, ax.get_ylim()[1] * 0.95, f"mean {avg_cls:.1%}  " if right else f"  mean {avg_cls:.1%}",
        color=INK2, fontsize=9, va="top", ha="right" if right else "left")
ax.xaxis.set_major_formatter(pct)
ax.set(title="Distribution of per-species accuracy", xlabel="Accuracy on that species",
       ylabel="Number of species")
save(fig, "05_per_class_distribution.png"); plt.show()

In [ ]:
# The 20 species the model struggles with most
worst = per.dropna().sort_values(["accuracy", "n_test"], ascending=[True, False]).head(20)[::-1]
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(worst.species, worst.accuracy, color=ORANGE, height=0.7)
for y, r in enumerate(worst.itertuples()):
    ax.text(r.accuracy + 0.01, y, f"{r.correct}/{r.n_test}", va="center", color=INK2, fontsize=9)
ax.set_xlim(0, 1.08)
ax.xaxis.set_major_formatter(pct)
ax.set(title="20 hardest species", xlabel="Test accuracy")
ax.grid(axis="y", visible=False)
save(fig, "06_hardest_species.png"); plt.show()

In [ ]:
# Accuracy by family: are look-alike groups really harder?
fam_acc = (per.groupby("family")
              .agg(species=("species", "size"), correct=("correct", "sum"), n=("n_test", "sum"))
              .query("species >= 4"))
fam_acc["accuracy"] = fam_acc.correct / fam_acc.n
fam_acc = fam_acc.sort_values("accuracy")

fig, ax = plt.subplots(figsize=(8, 0.32 * len(fam_acc) + 0.8))
ax.barh([f"{f} ({k})" for f, k in zip(fam_acc.index, fam_acc.species)], fam_acc.accuracy,
        color=BLUE, height=0.7)
ax.axvline(top1, color=INK2, linestyle="--", linewidth=1)
ax.text(top1, len(fam_acc) - 0.4, f"overall {top1:.1%}  ", color=INK2, fontsize=9, ha="right", va="bottom")
ax.set_xlim(0, 1.05)
ax.xaxis.set_major_formatter(pct)
ax.set(title="Test accuracy by bird family (number of species in brackets)", xlabel="Accuracy")
ax.grid(axis="y", visible=False)
save(fig, "07_family_accuracy.png"); plt.show()

### Confusion matrix

Row = the true species, column = what the model guessed. A perfect model is a single dark diagonal line.
Species are numbered roughly by family, so dark squares near the diagonal mean mix-ups
*within* a family (e.g. one warbler for another).

In [ ]:
cm = confusion_matrix(t, p, labels=range(num_classes))
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(cm_norm, cmap=BLUES, vmin=0, vmax=1, interpolation="nearest")
ax.grid(False)
ax.set(title="Confusion matrix (row-normalised)", xlabel="Predicted species (label)", ylabel="True species (label)")
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, format=pct)
cb.outline.set_visible(False)
save(fig, "08_confusion_matrix.png"); plt.show()

In [ ]:
# The most common specific mix-ups
off = cm.copy()
np.fill_diagonal(off, 0)
order = np.argsort(off.ravel())[::-1][:15]
pairs = [(i // num_classes, i % num_classes) for i in order if off.ravel()[i] > 0]
labels = [f"{CLASS_NAMES[a]}  →  {CLASS_NAMES[b]}" for a, b in pairs][::-1]
counts = [off[a, b] for a, b in pairs][::-1]

fig, ax = plt.subplots(figsize=(9, 0.34 * len(pairs) + 0.9))
ax.barh(labels, counts, color=ORANGE, height=0.7)
ax.set(title="Most frequent confusions (true → predicted)", xlabel="Test images")
ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax.grid(axis="y", visible=False)
save(fig, "09_top_confusions.png"); plt.show()

In [ ]:
# Confident mistakes: the test photo next to a training photo of the species it was mistaken for
first_train = {}
for name, label in train_raw:
    first_train.setdefault(label - offset, name)

wrong = np.where(p != t)[0]
wrong = wrong[np.argsort(-conf.numpy()[wrong])][:6]
if len(wrong):
    fig, axes = plt.subplots(len(wrong), 2, figsize=(7, 3.1 * len(wrong)), squeeze=False)
    for row, i in zip(axes, wrong):
        row[0].imshow(Image.open(test_ds.samples[i][0]).convert("RGB"))
        row[0].set_title(f"Test photo — actually: {CLASS_NAMES[t[i]]}", fontsize=9, loc="left", fontweight="normal")
        row[1].imshow(Image.open(find_image(TRAIN_DIR, first_train[p[i]])).convert("RGB"))
        row[1].set_title(f"Guessed: {CLASS_NAMES[p[i]]} ({conf[i]:.0%} sure)", fontsize=9, loc="left",
                         fontweight="normal", color=ORANGE)
        for ax in row:
            ax.axis("off")
    fig.tight_layout()
    save(fig, "10_confident_mistakes.png"); plt.show()

## 4. Comparing runs (optional)

Train other configurations into their own folders, e.g.
`python train.py ... --model resnet50 --out-dir checkpoints_resnet50` or
`python train.py ... --no-pretrained --out-dir checkpoints_scratch`, then list them here.

In [ ]:
RUNS = {  # label -> folder containing history.csv
    "ConvNeXt-Tiny (pretrained)": "checkpoints",
    # "ResNet-50 (pretrained)": "checkpoints_resnet50",
    # "ConvNeXt-Tiny (from scratch)": "checkpoints_scratch",
}
runs = {k: pd.read_csv(os.path.join(v, "history.csv")) for k, v in RUNS.items()
        if os.path.exists(os.path.join(v, "history.csv"))}

fig, ax = plt.subplots(figsize=(8, 4))
for (label, h), color in zip(runs.items(), [BLUE, ORANGE, AQUA]):  # max 3 runs per chart
    ax.plot(h.epoch, h.val_avg_class, color=color, label=label)
ax.yaxis.set_major_formatter(pct)
ax.set(title="Validation avg per-class accuracy by run", xlabel="Epoch", ylabel="Accuracy")
if len(runs) > 1:
    ax.legend(loc="lower right")
save(fig, "11_run_comparison.png"); plt.show()